In [ ]:
import asyncio
import logging
import os
from pprint import pprint
from typing import Union

import httpx
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import AsyncOpenAI, OpenAI
from parsel import Selector


In [6]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

In [7]:
# Check the key
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [8]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages

[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [9]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
pprint(response.choices[0].message.content)

('Hi there! Nice to meet you. I’m here to help with questions, writing, '
 'coding, brainstorming, learning new things, and more. If you tell me a bit '
 'about what you’re into or what you’d like to do, I’ll tailor my replies.\n'
 '\n'
 'A few quick ideas to start:\n'
 '- Answer a question you have\n'
 '- Help plan a project or write something (email, story, resume)\n'
 '- Do a quick mini-lesson on a topic\n'
 '- Brainstorm ideas or practice a skill with a short exercise\n'
 '\n'
 'What would you like to do first?')


In [10]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website: {page_context}.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
"""

In [11]:
# TODO: Add semaphore and rate limiting - Understand and implement
# TODO: Add a generic xpath to extract all the texts from html page
# TODO: Add a scraping logic based on page type - Httpx + Parsel or Selenium / Playwright + Parsel ?

async def make_request(client: httpx.AsyncClient, url: str):
    response = await client.get(url=url)
    # logging.warning(f'Msg: Response recieved for {url} | Status: {response.status_code}')
    response.raise_for_status()
    return response

async def process_urls(urls: list[str]) -> list[str]:
    async with httpx.AsyncClient() as client:
        responses = await asyncio.gather(*(make_request(client, url) for url in urls))
        return [x.text for x in responses]


async def scrape_urls(urls: list[str]) -> list[str]:
    html_responses = await process_urls(urls)
    html_parsed = []
    for html_text in html_responses:
        selector = Selector(text=html_text)

        paragraphs = selector.xpath(
            '//div[contains('
            'concat(" ", normalize-space(@class), " "), '
            '" wp-block-cover__inner-container "'
            ')]//p'
        )

        paragraph_texts = []
        for paragraph in paragraphs:
            text_parts = paragraph.xpath(".//text()").getall()
            text = " ".join(" ".join(text_parts).split())
            if text:
                paragraph_texts.append(text)

        html_parsed.append('\n'.join(paragraph_texts))
    return html_parsed


def format_llm_prompt(user_content: str):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix.format(page_context=user_content)}
    ]
    return messages


async def make_llm_request(client: AsyncOpenAI, user_content: str):
    llm_response = await client.chat.completions.create(model="gpt-4.1-mini", messages=format_llm_prompt(user_content))
    logging.info(f'Msg: Response recieved from llm for user content: {user_content}')
    return llm_response


async def call_llm(user_contents: list[str]):
    async with AsyncOpenAI() as client:
        llm_responses = await asyncio.gather(*(make_llm_request(client=client, user_content=user_content) for user_content in user_contents))
        return llm_responses

async def summarize(urls):
    page_texts = await scrape_urls(urls=urls)
    llm_responses = await call_llm(user_contents=page_texts)
    for llm_response in llm_responses:
        print(llm_response.choices[0].message.content)
        print('\n\n')

        

In [12]:
urls = ["https://edwarddonner.com", "https://edition.cnn.com/", "https://www.anthropic.com/"]
await summarize(urls=urls)

**Ed’s Corner of the Internet: Where Code Meets Ego**

Meet Ed: a code-loving, LLM-obsessed guy who dabbles in amateur electronic music (brace yourself) and pretends to understand Hacker News like a boss. Former JPMorgan bigwig turned AI startup maestro, Ed now runs Nebula.io and brags about his previous startup’s acquisition like it’s a trophy.

Tired of annoying his friends with endless LLM rants, he shoved that energy into Udemy courses, which surprisingly turned into global hits with over a million students. Welcome to Ed’s world—lecture-happy, nerdy, slightly self-absorbed, but hey, the courses are top-notch.

Oh, and if you want to chat with Ed’s digital twin, he’s got one of those too because why not?



Looks like someone forgot to add the website content. Did the internet eat it, or is this a sneaky prank? Either way, no juicy details to summarize here!



Well, this website seems to be a shining beacon of absolutely nothing. Zero content, zero announcements, zero news—just th